In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from pathlib import Path

#below is the path where all 9 datasets are stored there in this folder

#the r tells the python dont treat backlashes as escape characters
raw_data_dir = Path(r"D:\Projects\Retails-Analytics-Data-Warehouse\data\raw")

#converting all our csvs to list so that we run loop over it
#glob is used to tell the script go into this folder and find everything that 
#has .csv
csv_files = list(raw_data_dir.glob("*.csv"))

print(f"Found {len(csv_files)} csv files.\n")

for file in csv_files:

    df = pd.read_csv(file)

    print("="*50)
    print(f"File: {file.name}")
    print("="*50)

    print(f"Rows: {df.shape[0]:,}")
    print(f"columns: {df.shape[1]}")

    print("\ncolumns")

    print(df.columns.tolist())

    print("\n Data Types")
    print(df.dtypes)

    print("\nMissing Values: ")

    print(df.isnull().sum())

    print("\nDuplicate Rows: ")
    print(df.duplicated().sum())

    print()

Found 9 csv files.

File: olist_customers_dataset.csv
Rows: 99,441
columns: 5

columns
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

 Data Types
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing Values: 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows: 
0

File: olist_geolocation_dataset.csv
Rows: 1,000,163
columns: 5

columns
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

 Data Types
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Mi

In [4]:
#naming the datasets

customers = pd.read_csv(
    raw_data_dir / "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    raw_data_dir / "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    raw_data_dir / "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    raw_data_dir / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    raw_data_dir / "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    raw_data_dir / "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    raw_data_dir / "olist_sellers_dataset.csv"
)

geolocation = pd.read_csv(
    raw_data_dir / "olist_geolocation_dataset.csv"
)

category_translation = pd.read_csv(
    raw_data_dir / "product_category_name_translation.csv"
)

print("All 9 datasets loaded successfully.")

All 9 datasets loaded successfully.


In [4]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category Translation:", category_translation.shape)

Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (1000163, 5)
Category Translation: (71, 2)


#### Customer Table transformation

In [10]:
#there are no missing values no duplicates record in this table so we are going to just  standardiez column names and data types so then it is easy to 
#import it to database

customers.columns = (
    customers.columns
    .str.strip()
    .str.lower()
)


#comverted to one format
print(customers.columns.tolist())

text_columns = [
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
]


#this remove the extraspaces in the columns name 
for column in text_columns:
    customers[column] = customers[column].str.strip()


print("Rows:", len(customers))
print("Duplicate rows:", customers.duplicated().sum())
print("Missing values:")
print(customers.isnull().sum())

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Rows: 99441
Duplicate rows: 0
Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [12]:
#checking it for our sql format 



print("CUSTOMER TABLE - MYSQL COMPATIBILITY CHECK")


print("\nColumns:")
print(customers.columns.tolist())

print("\nPandas Data Types:")
print(customers.dtypes)

CUSTOMER TABLE - MYSQL COMPATIBILITY CHECK

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Pandas Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object


In [13]:
#as for below column we put char(2) there in the database so lets chekc it if it exactly contians the 2 char

print("\nCustomer state lengths:")
print(customers["customer_state"].str.len().value_counts())


Customer state lengths:
customer_state
2    99441
Name: count, dtype: int64


In [15]:
print("ZIP prefix data type:")
print(customers["customer_zip_code_prefix"].dtype)

print("\nZIP prefix range:")
print("Minimum:", customers["customer_zip_code_prefix"].min())
print("Maximum:", customers["customer_zip_code_prefix"].max())

#so there is no negative or enormous value as suggest by this output

ZIP prefix data type:
int64

ZIP prefix range:
Minimum: 1003
Maximum: 99990


In [17]:
#saved it t processd data folder

customers.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\customers.csv",
    index=False
)

print("Customer data saved successfully")

Customer data saved successfully


### Orders Transformation

In [18]:
orders.dtypes

#look even the dates are object we are goning to convert it into datetime

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [20]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])


orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [21]:
print(orders[date_columns].isnull().sum())

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [25]:
#standardize the orders columns

orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
)

print(orders.columns.tolist())


#removing if there is accidental whitespaces
text_columns = [
    "order_id",
    "customer_id",
    "order_status"
]

for column in text_columns:
    orders[column] = orders[column].str.strip()

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [26]:
print("Rows:", len(orders))
print("Duplicate rows:", orders.duplicated().sum())

print("\nMissing values:")
print(orders.isnull().sum())

print("\nData types:")
print(orders.dtypes)

Rows: 99441
Duplicate rows: 0

Missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Data types:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


if we look at the above date columns then there are some missing values but it does not mean we should remve it or replace it as 
there is like column which is order_delivered_customer_date having 2965 null values but we cant remove that maybe that means the order is not delieverd yet so 
we have not recorded the date like for some order may be cancelled before so no date is recorded 

In [27]:
orders.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\orders.csv",
    index=False
)

print("Orders data saved successfully")

Orders data saved successfully


### Order_Items_table

In [29]:
order_items.dtypes

#these order_id are stored as the collection of int and string like
#"87285b5f6b..."

#while the date column need to be changed


order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [30]:
order_items.columns = (
    order_items.columns
    .str.strip()
    .str.lower()
)

print(order_items.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [31]:
#convertning the data type object to datetime

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)

In [32]:
text_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

for column in text_columns:
    order_items[column] = order_items[column].str.strip()

In [33]:
print("=" * 60)
print("ORDER ITEMS VALIDATION")
print("=" * 60)

print("Rows:", len(order_items))

print("\nDuplicate rows:")
print(order_items.duplicated().sum())

print("\nMissing values:")
print(order_items.isnull().sum())

print("\nData types:")
print(order_items.dtypes)

ORDER ITEMS VALIDATION
Rows: 112650

Duplicate rows:
0

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Data types:
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object


In [34]:
#checking if our composite key is ok

print(
    "Duplicate composite keys:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Duplicate composite keys: 0


In [35]:
order_items.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\order_items.csv",
    index=False
)

print("Order Items data saved successfully")

Order Items data saved successfully


### Payments table

In [38]:
#it also has a composite key and there is 0 duplicate so we are good with it


display(payments.head())
display(payments.dtypes)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [39]:
#standardize the names 
payments.columns = (
    payments.columns
    .str.strip()
    .str.lower()
)

print(payments.columns.tolist())

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


In [40]:
#strip the text columns only

#removing accidental whitespaces
text_columns = [
    "order_id",
    "payment_type"
]

for column in text_columns:
    payments[column] = payments[column].str.strip()

In [41]:
print(payments["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [44]:
#lets check those undefined records

payments[payments["payment_type"] == "not_defined"]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [42]:
print("Payment installments:")
print(payments["payment_installments"].describe())

print("\nPayment value:")
print(payments["payment_value"].describe())

Payment installments:
count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

Payment value:
count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64


In [45]:
#lets investigte why installements = 0?
payments[payments["payment_installments"] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [43]:
print("=" * 60)
print("PAYMENTS VALIDATION")
print("=" * 60)

print("Rows:", len(payments))
print("Duplicate rows:", payments.duplicated().sum())

print("\nMissing values:")
print(payments.isnull().sum())

print("\nDuplicate composite keys:")
print(
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print("\nData types:")
print(payments.dtypes)

PAYMENTS VALIDATION
Rows: 103886
Duplicate rows: 0

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Duplicate composite keys:
0

Data types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object


In [46]:
#final validaiton


print("=" * 60)
print("PAYMENTS FINAL VALIDATION")
print("=" * 60)

print("Rows:", len(payments))
print("Duplicate rows:", payments.duplicated().sum())

print("\nMissing values:")
print(payments.isnull().sum())

print("\nDuplicate composite keys:")
print(
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print("\nData types:")
print(payments.dtypes)

PAYMENTS FINAL VALIDATION
Rows: 103886
Duplicate rows: 0

Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Duplicate composite keys:
0

Data types:
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object


In [47]:
payments.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\payments.csv",
    index=False
)

print("Payments data saved successfully")

Payments data saved successfully


### Reviews Table

In [52]:
display(reviews.head())
display(reviews.dtypes)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

In [53]:
reviews.columns = (
    reviews.columns
    .str.strip()
    .str.lower()
)

print(reviews.columns.tolist())

['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


In [54]:
text_id_columns = [
    "review_id",
    "order_id"
]

for column in text_id_columns:
    reviews[column] = reviews[column].str.strip()

In [55]:
date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in date_columns:
    reviews[column] = pd.to_datetime(reviews[column])

In [57]:
reviews.dtypes

review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object

In [ ]:
#the review_comment_title and review_comment_message are null but you know not all customers leave reviews they sometime
#submit the review score without leaving comment or title

In [58]:
#displaying the reiew score range

print("Review score values:")
print(sorted(reviews["review_score"].unique()))

Review score values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [59]:

print("REVIEWS FINAL VALIDATION")

print("Rows:", len(reviews))

print("\nDuplicate complete rows:")
print(reviews.duplicated().sum())

print("\nMissing values:")
print(reviews.isnull().sum())

print("\nReview score values:")
print(sorted(reviews["review_score"].unique()))

print("\nDuplicate (review_id, order_id) combinations:")
print(
    reviews.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

print("\nData types:")
print(reviews.dtypes)

REVIEWS FINAL VALIDATION
Rows: 99224

Duplicate complete rows:
0

Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Review score values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Duplicate (review_id, order_id) combinations:
0

Data types:
review_id                          object
order_id                           object
review_score                        int64
review_comment_title               object
review_comment_message             object
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


In [60]:
reviews.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\reviews.csv",
    index=False
)

print("Reviews data saved successfully")

Reviews data saved successfully


### Product table

In [62]:
display(products.head())
display(products.dtypes)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [64]:
products.columns = (
    products.columns
    .str.strip()
    .str.lower()
)
print(products.columns.tolist())

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [66]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

In [67]:
products["product_id"] = products["product_id"].str.strip()

In [69]:
#investiagting the 610 missing products

missing_product_info = products[
    products["product_category_name"].isnull()
]

print("Products with missing category:", len(missing_product_info))

print("\nMissing values in these products:")
print(missing_product_info.isnull().sum())

Products with missing category: 610

Missing values in these products:
product_id                      0
product_category_name         610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


In [70]:
missing_product_info = products[
    products["product_category_name"].isnull()
]

print("Products with missing category:", len(missing_product_info))

print("\nMissing values in these products:")
print(missing_product_info.isnull().sum())

Products with missing category: 610

Missing values in these products:
product_id                      0
product_category_name         610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


In [71]:
print(
    products[
        products["product_weight_g"].isnull()
    ][[
        "product_id",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]]
)

                             product_id  product_weight_g  product_length_cm  \
8578   09ff539a621711667c43eba6a3bd8466               NaN                NaN   
18851  5eb564652db742ff8f28759cd8d2652a               NaN                NaN   

       product_height_cm  product_width_cm  
8578                 NaN               NaN  
18851                NaN               NaN  


In [72]:
#so from this cell we foun out we have like 73 unique categoires in product table 
#but our translaiton table contains only 71 categories we already discovered this in data profiing phase


print("Unique product categories:")
print(products["product_category_name"].nunique())

print("\nTop product categories:")
print(products["product_category_name"].value_counts().head(20))

Unique product categories:
73

Top product categories:
product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
eletronicos                           517
construcao_ferramentas_construcao     400
eletrodomesticos                      370
Name: count, dtype: int64


In [73]:
#so from below we discovered 2 translaiton are missed

translated_categories = set(
    category_translation["product_category_name"]
)

product_categories = set(
    products["product_category_name"].dropna()
)

missing_translations = product_categories - translated_categories

print("Product categories:", len(product_categories))
print("Translated categories:", len(translated_categories))
print("Missing translations:", len(missing_translations))

print("\nCategories without translation:")
print(sorted(missing_translations))

Product categories: 73
Translated categories: 71
Missing translations: 2

Categories without translation:
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [74]:
#final validation

print("=" * 60)
print("PRODUCTS FINAL VALIDATION")
print("=" * 60)

print("Rows:", len(products))

print("\nDuplicate complete rows:")
print(products.duplicated().sum())

print("\nDuplicate product IDs:")
print(products["product_id"].duplicated().sum())

print("\nMissing values:")
print(products.isnull().sum())

print("\nData types:")
print(products.dtypes)

PRODUCTS FINAL VALIDATION
Rows: 32951

Duplicate complete rows:
0

Duplicate product IDs:
0

Missing values:
product_id                      0
product_category_name         610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Data types:
product_id                     object
product_category_name          object
product_name_length           float64
product_description_length    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


In [75]:
products.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\products.csv",
    index=False
)

print("Products data saved successfully")

Products data saved successfully


### Seller table

In [6]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [8]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [5]:
sellers.columns = (
    sellers.columns
    .str.strip()
    .str.lower()
)

In [9]:
text_columns = [
    "seller_id",
    "seller_city",
    "seller_state"
]

for column in text_columns:
    sellers[column] = sellers[column].str.strip()

In [10]:
print("=" * 60)
print("SELLERS VALIDATION")
print("=" * 60)

print("Rows:", len(sellers))

print("\nDuplicate complete rows:")
print(sellers.duplicated().sum())

print("\nDuplicate seller IDs:")
print(sellers["seller_id"].duplicated().sum())

print("\nMissing values:")
print(sellers.isnull().sum())

print("\nData types:")
print(sellers.dtypes)

SELLERS VALIDATION
Rows: 3095

Duplicate complete rows:
0

Duplicate seller IDs:
0

Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Data types:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object


In [11]:
sellers.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\sellers.csv",
    index=False
)

print("Sellers data saved successfully")

Sellers data saved successfully


### Geolocation table

In [13]:
geolocation = pd.read_csv(
    raw_data_dir / "olist_geolocation_dataset.csv"
)


In [14]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [15]:
geolocation.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

In [16]:
geolocation.columns = (
    geolocation.columns
    .str.strip()
    .str.lower()
)

In [17]:
text_columns = [
    "geolocation_city",
    "geolocation_state"
]

for column in text_columns:
    geolocation[column] = geolocation[column].str.strip()

In [19]:

print("GEOLOCATION VALIDATION")


print("Rows:", len(geolocation))

#so why this much duplicate itis because of the zip so zip is repeated but inside that zip the coordniates are like diff
#but we can find state of a customer or a seller if we match it by zip_code of a cusomter as that is a matching column


#lat and longt are different 

print("\nDuplicate complete rows:")
print(geolocation.duplicated().sum())

print("\nMissing values:")
print(geolocation.isnull().sum())

print("\nData types:")
print(geolocation.dtypes)

GEOLOCATION VALIDATION
Rows: 1000163

Duplicate complete rows:
0          False
1          False
2          False
3          False
4          False
           ...  
1000158    False
1000159     True
1000160     True
1000161    False
1000162     True
Length: 1000163, dtype: bool

Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object


In [20]:
duplicate_rows = geolocation[
    geolocation.duplicated(keep=False)
].sort_values(
    by=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
)

print("Total rows involved in exact duplicates:", len(duplicate_rows))

print("\nNumber of exact duplicate groups:",
      duplicate_rows.drop_duplicates().shape[0])

print("\nSample exact duplicates:")
print(duplicate_rows.head(30))

Total rows involved in exact duplicates: 390005

Number of exact duplicate groups: 128174

Sample exact duplicates:
      geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
519                          1001       -23.551337       -46.634027   
583                          1001       -23.551337       -46.634027   
818                          1001       -23.551337       -46.634027   
206                          1001       -23.550498       -46.634338   
429                          1001       -23.550498       -46.634338   
596                          1001       -23.550498       -46.634338   
639                          1001       -23.550498       -46.634338   
771                          1001       -23.550498       -46.634338   
912                          1001       -23.550498       -46.634338   
985                          1001       -23.550498       -46.634338   
1062                         1001       -23.550498       -46.634338   
851                          100

In [21]:
geolocation = geolocation.drop_duplicates().copy()

In [22]:

#we just dropped the exact dupicated as diaplyed in above cell keeping only one copy

print("Rows after removing exact duplicates:", len(geolocation))
print("Duplicate rows remaining:", geolocation.duplicated().sum())


Rows after removing exact duplicates: 738332
Duplicate rows remaining: 0


In [23]:
geolocation.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\geolocation.csv",
    index=False
)

print("Geolocation data saved successfully")

Geolocation data saved successfully


In [24]:
category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [25]:
category_translation.dtypes

product_category_name            object
product_category_name_english    object
dtype: object

In [27]:
category_translation.columns = (
    category_translation.columns
    .str.strip()
    .str.lower()
)

In [28]:
for column in category_translation.columns:
    category_translation[column] = (
        category_translation[column].str.strip()
    )

In [29]:

print("CATEGORY TRANSLATION VALIDATION")


print("Rows:", len(category_translation))

print("\nDuplicate complete rows:")
print(category_translation.duplicated().sum())

print("\nMissing values:")
print(category_translation.isnull().sum())

print("\nDuplicate Portuguese categories:")
print(
    category_translation["product_category_name"]
    .duplicated()
    .sum()
)

print("\nDuplicate English categories:")
print(
    category_translation["product_category_name_english"]
    .duplicated()
    .sum()
)

print("\nData types:")
print(category_translation.dtypes)




#total produts are 73 we have translation of 71 two missing but we are not adding the data it dpeened on business value if required then we can easily 
#add it as we know those missing values

CATEGORY TRANSLATION VALIDATION
Rows: 71

Duplicate complete rows:
0

Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64

Duplicate Portuguese categories:
0

Duplicate English categories:
0

Data types:
product_category_name            object
product_category_name_english    object
dtype: object


In [30]:
category_translation.to_csv(
    r"D:\Projects\Retails-Analytics-Data-Warehouse\data\processed\category_translation.csv",
    index=False
)

print("Category translation data saved successfully")

Category translation data saved successfully
